## Step 0: Import libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 200)

## Step 1: Create a synthetic dataset

In [ ]:
rng = np.random.default_rng(42)
n = 100
age = rng.integers(22, 60, n).astype(float)
salary = np.round(25000 + age * 900 + rng.normal(0, 6000, n), -2)
df = pd.DataFrame({
    "Country": rng.choice(["France", "Spain", "Germany"], n),
    "Age": age,
    "Salary": salary,
    "Purchased": np.where(salary + rng.normal(0, 8000, n) > 55000, "Yes", "No"),
})
df.loc[rng.choice(n, 5, replace=False), "Age"] = np.nan
df.loc[rng.choice(n, 5, replace=False), "Salary"] = np.nan
df.head(10)

## Step 2: Fill missing values (median) so the scalers can run

In [ ]:
num_cols = ["Age", "Salary"]
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[num_cols].isna().sum()

## Step 3: Apply `StandardScaler` and `MinMaxScaler`

In [ ]:
std_scaled = pd.DataFrame(StandardScaler().fit_transform(df[num_cols]), columns=num_cols)
mm_scaled  = pd.DataFrame(MinMaxScaler().fit_transform(df[num_cols]),  columns=num_cols)

## Step 4: Compare the distributions

In [ ]:
stats = ["mean", "std", "min", "max"]
print("=== Original ===");                         print(df[num_cols].describe().loc[stats].round(3))
print("\n=== StandardScaler (mean ≈ 0, std ≈ 1) ===");  print(std_scaled.describe().loc[stats].round(3))
print("\n=== MinMaxScaler (range [0, 1]) ===");        print(mm_scaled.describe().loc[stats].round(3))

## Step 5: Train/Test split – 80% / 20%

In [ ]:
X = pd.get_dummies(df.drop(columns=["Purchased"]), columns=["Country"], dtype=int)
y = df["Purchased"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Total rows: {len(X)} | Train: {X_train.shape} | Test: {X_test.shape}")

## Step 6: Correct practice – fit the scaler on TRAIN only
Fitting on the training set and only *transforming* the test set prevents **data leakage**.

In [ ]:
scaler = StandardScaler()
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols]  = scaler.transform(X_test[num_cols])

print("Scaled TRAIN stats (mean ≈ 0, std ≈ 1):")
print(X_train_s[num_cols].describe().loc[["mean", "std"]].round(3))
print("\nScaled TEST stats (close to, but not exactly, 0/1 – as expected):")
print(X_test_s[num_cols].describe().loc[["mean", "std"]].round(3))